# 00 - Drive Setup and File Check

Creates the Google Drive project structure for the Turkish Legal RAG project and checks whether the required raw files are present. This notebook does not run fine-tuning.

In [ ]:
from pathlib import Path
import json

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running in Google Colab; using local filesystem paths.')

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
DRIVE_ROOT

In [ ]:
PROJECT_DIRS = [
    'data/raw',
    'data/processed',
    'data/benchmark',
    'data/custom_docs',
    'indexes/dense',
    'indexes/bm25',
    'indexes/hybrid',
    'indexes/reranker',
    'models/adapters',
    'models/embedding_tuned',
    'models/reranker_tuned',
    'outputs/retrieval_eval',
    'outputs/generation_eval',
    'outputs/ablation',
    'outputs/predictions',
    'reports',
    'notebooks',
    'src',
]

for rel_path in PROJECT_DIRS:
    (DRIVE_ROOT / rel_path).mkdir(parents=True, exist_ok=True)

custom_hint = DRIVE_ROOT / 'data/custom_docs/put_custom_documents_here.txt'
if not custom_hint.exists():
    custom_hint.write_text('Put instructor-provided custom documents in this folder for the later custom ingestion pipeline.\n', encoding='utf-8')

print(f'Created/verified project structure under: {DRIVE_ROOT}')

In [ ]:
config = {
    'project_name': 'turkish_legal_rag',
    'drive_root': str(DRIVE_ROOT),
    'corpus_version': 'v3',
    'raw_curated_csv': 'data/raw/legal_documents_curated.csv',
    'old_baseline_eval': 'data/benchmark/evaluation_results_baseline_old.csv',
    'processed_dir': 'data/processed',
    'reports_dir': 'reports',
    'main_law_corpus_csv': 'data/processed/legal_main_law_corpus_v3.csv',
    'main_law_corpus_jsonl': 'data/processed/legal_main_law_corpus_v3.jsonl',
    'qa_auxiliary_csv': 'data/processed/legal_qa_auxiliary_v3.csv',
    'rejected_review_csv': 'data/processed/legal_rejected_review_v3.csv',
    'normalization_report_json': 'reports/normalization_report_v3.json',
    'benchmark_csv': 'data/benchmark/gold_benchmark_v1.csv',
    'official_index_root': 'indexes/official_law_v3',
    'external_dataset': {
        'root': 'data/external',
        'corpus': 'data/external/corpus.jsonl',
        'llm_sft': 'data/external/llm.jsonl',
        'embedding_pairs': 'data/external/embedding.jsonl',
        'reranker_pairs': 'data/external/reranker.jsonl',
        'secondary_gold_benchmark': 'data/external/gold_benchmark.json',
        'secondary_rag_eval': 'data/external/rag_eval.json',
    },
    'retrieval_text_field': 'retrieval_text',
    'generation_text_field': 'generation_text',
    'citation_field': 'citation_label',
    'base_llm_model': 'google/gemma-2-2b-it',
    'embedding_models': {
        'default': 'BAAI/bge-m3',
        'alternative': 'intfloat/multilingual-e5-base',
    },
    'retrieval_defaults': {
        'top_k_retrieval': 30,
        'top_k_context': 5,
        'bm25_weight': 0.45,
        'dense_weight': 0.55,
    },
    'normalization': {
        'official_source_domain': 'mevzuat.gov.tr',
        'valid_quality_flags': ['valid_article', 'short_but_valid'],
        'review_quality_flags': [
            'empty_or_too_short',
            'metadata_only',
            'amendment_fragment',
            'table_fragment',
            'duplicate_candidate',
            'duplicate_conflict',
        ],
    },
}

config_path = DRIVE_ROOT / 'project_config.json'
config_path.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Wrote config: {config_path}')

In [ ]:
required_files = {
    'curated corpus': DRIVE_ROOT / config['raw_curated_csv'],
    'old baseline eval': DRIVE_ROOT / config['old_baseline_eval'],
}

status_rows = []
for label, path in required_files.items():
    status_rows.append({
        'file': label,
        'path': str(path),
        'exists': path.exists(),
        'size_mb': round(path.stat().st_size / (1024 * 1024), 3) if path.exists() else None,
    })

status_rows

If `legal_documents_curated.csv` is missing, upload it to `data/raw/`. If `evaluation_results_baseline_old.csv` is missing, upload it to `data/benchmark/`. Then continue with notebook `01_normalize_curated_corpus_v3.ipynb`.